# Introductory Statistics – Part 4: Power, p-values, and the t-Distribution

Welcome to Part 4. We deepen our understanding of hypothesis testing by examining **statistical power**, **p-values**, and inference when $\sigma$ is unknown — the most common real-world situation.

**By the end of this notebook you will be able to:**

- Define statistical power and explain what affects it
- Calculate the required sample size to achieve a desired power
- Compute and correctly interpret a p-value
- Explain the properties of the $t$-distribution and when to use it
- Construct a $t$-based confidence interval and perform a one-sample $t$-test

**Topics covered:**

13. Power of a test and choosing sample size  
14. p-values: computation and interpretation  
15. Inference about $\mu$ with unknown $\sigma$: the $t$-distribution

> **Prerequisite:** Part 3 – Sampling and Statistical Inference.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, t

sns.set(style="whitegrid")
np.random.seed(42)

---
## 13. Power of a Test and Choosing Sample Size

### What is statistical power?

**Power** = $P(\text{reject } H_0 \mid H_1 \text{ is true}) = 1 - \beta$

It is the probability of correctly detecting a real effect. Power close to 1 is desirable.

### What affects power?

| Factor | Effect on power |
|---|---|
| True mean farther from $\mu_0$ (larger effect size) | ↑ increases |
| Larger sample size $n$ | ↑ increases |
| Smaller $\sigma$ | ↑ increases |
| Larger significance level $\alpha$ | ↑ increases (but raises Type I error) |

### Power for a one-sided z-test

For $H_0: \mu = \mu_0$ vs $H_1: \mu > \mu_0$, the power at true mean $\mu_1$ is:

$$\text{Power}(\mu_1) = 1 - \Phi\!\left(z_{1-\alpha} - \frac{\mu_1 - \mu_0}{\sigma/\sqrt{n}}\right)$$

where $\Phi$ is the standard normal CDF.

In [ ]:
# Power curve: H0: μ = 100 vs H1: μ > 100
mu0   = 100
sigma = 10
alpha = 0.05
z_crit = norm.ppf(1 - alpha)

mu_values  = np.linspace(100, 120, 300)
power_vals = []

for n_val in [10, 30, 60]:
    pw = [1 - norm.cdf(z_crit - (mu - mu0) / (sigma / np.sqrt(n_val)))
          for mu in mu_values]
    power_vals.append(pw)

plt.figure(figsize=(8, 4))
for pw, n_val in zip(power_vals, [10, 30, 60]):
    plt.plot(mu_values, pw, linewidth=2, label=f"$n = {n_val}$")

plt.axhline(0.80, color="gray", linestyle="--", alpha=0.7, label="80% power target")
plt.axhline(alpha, color="lightcoral", linestyle="--", alpha=0.7,
            label=f"$\\alpha = {alpha}$ (minimum power at $\\mu_0$)")
plt.title("Power Curves for One-Sample z-Test ($\\sigma=10,\\; \\alpha=0.05$)")
plt.xlabel("True Mean $\\mu_1$")
plt.ylabel("Power")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

### 13.1 Sample Size for Desired Power

For a one-sided z-test, the sample size needed to achieve power $1 - \beta$ is:

$$n = \left(\frac{z_{1-\alpha} + z_{1-\beta}}{(\mu_1 - \mu_0)/\sigma}\right)^2$$

The quantity $(\mu_1 - \mu_0)/\sigma$ is the **standardised effect size** (similar to Cohen's $d$). Larger effects require smaller samples.

In [ ]:
# Required sample size for 80% power
mu0           = 100
mu1           = 105    # true mean under H1
sigma         = 10
alpha         = 0.05
power_target  = 0.80

z_alpha  = norm.ppf(1 - alpha)
z_beta   = norm.ppf(power_target)
effect   = (mu1 - mu0) / sigma   # standardised effect size

n_required = ((z_alpha + z_beta) / effect) ** 2

print(f"H0: μ = {mu0},  H1: μ = {mu1},  σ = {sigma}")
print(f"Standardised effect size: {effect:.2f}")
print(f"z_α = {z_alpha:.3f},  z_β = {z_beta:.3f}")
print(f"Required n for {power_target*100:.0f}% power: {np.ceil(n_required):.0f}")

---
## 14. p-values

### Definition

> A **p-value** is the probability, *assuming $H_0$ is true*, of obtaining a test statistic at least as extreme as the one observed.

### How to interpret a p-value

| p-value | Interpretation |
|---|---|
| Small (e.g., $< 0.05$) | Strong evidence against $H_0$; reject $H_0$ |
| Large (e.g., $\geq 0.05$) | Weak evidence against $H_0$; fail to reject $H_0$ |

### Common misconceptions

- ❌ "The p-value is the probability that $H_0$ is true." → Wrong.
- ❌ "A large p-value proves $H_0$." → Wrong; it only means the data are *consistent* with $H_0$.
- ✅ The p-value measures how surprising the data are *if* $H_0$ were true.

In [ ]:
# One-sided z-test: H0: μ = 50 vs H1: μ > 50
mu0   = 50
sigma = 8
alpha = 0.05

np.random.seed(99)
data   = np.random.normal(loc=53, scale=8, size=40)
xbar   = data.mean()
n      = len(data)

z_stat  = (xbar - mu0) / (sigma / np.sqrt(n))
p_value = 1 - norm.cdf(z_stat)    # right-tail area

print(f"Sample mean (x̄): {xbar:.3f}")
print(f"z statistic:      {z_stat:.3f}")
print(f"p-value:          {p_value:.4f}")
print()
print("Decision at α = 0.05:",
      "Reject H₀" if p_value < alpha else "Fail to reject H₀")

In [ ]:
# Visualise the p-value as the tail area beyond the observed z

x_range = np.linspace(-4, 5, 400)
y_range = norm.pdf(x_range)

plt.figure(figsize=(8, 4))
plt.plot(x_range, y_range, color="black", linewidth=2)

# p-value region (right tail beyond observed z)
plt.fill_between(x_range, y_range, where=(x_range >= z_stat),
                 color="blue", alpha=0.35, label=f"p-value = {p_value:.4f}")

# Rejection region
z_crit = norm.ppf(1 - alpha)
plt.fill_between(x_range, y_range, where=(x_range >= z_crit),
                 color="red", alpha=0.35, label=f"Rejection region ($\\alpha={alpha}$)")

plt.axvline(z_stat, color="blue", linestyle="--",
            label=f"Observed $z = {z_stat:.2f}$")

plt.title("p-value Visualisation (One-Sided Test)")
plt.xlabel("$z$")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

---
## 15. Inference about $\mu$ with Unknown $\sigma$: the t-Distribution

In practice, $\sigma$ is almost never known. When we substitute the sample standard deviation $s$, the test statistic follows a **$t$-distribution** rather than a standard normal.

$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}} \sim t_{n-1}$$

### Properties of the $t$-distribution

- Symmetric and bell-shaped, like $N(0,1)$, but with **heavier tails**
- Parameterised by **degrees of freedom** $df = n - 1$
- As $n \to \infty$ (equivalently $df \to \infty$), $t_{df} \to N(0,1)$

### Assumptions

1. Data come from a **random sample**
2. Observations are **independent**
3. Population is **approximately normal** (critical for small $n$; less important for $n \geq 30$ by the CLT)

### $t$-based confidence interval for $\mu$

$$\bar{x} \pm t^*_{n-1} \cdot \frac{s}{\sqrt{n}}$$

where $t^*_{n-1}$ is the critical value from the $t_{n-1}$ distribution.

In [ ]:
# Compare t-distributions with different df to the standard normal

x_range = np.linspace(-4, 4, 400)

plt.figure(figsize=(7, 4))
plt.plot(x_range, norm.pdf(x_range), color="black",
         linewidth=2, linestyle="--", label="$N(0,1)$")

for df_val, color in [(2, "red"), (5, "orange"), (30, "steelblue")]:
    plt.plot(x_range, t.pdf(x_range, df_val),
             color=color, linewidth=1.5, label=f"$t_{{df={df_val}}}$")

plt.title("$t$-Distributions vs Standard Normal")
plt.xlabel("$x$")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

print("Note: smaller df → heavier tails → wider CIs and larger critical values.")

In [ ]:
# t-based 95% confidence interval for a mean (σ unknown)
np.random.seed(7)
data   = np.random.normal(loc=75, scale=12, size=20)
xbar   = data.mean()
s      = data.std(ddof=1)    # sample std dev
n      = len(data)
df     = n - 1
t_crit = t.ppf(0.975, df)    # two-sided 95%

se    = s / np.sqrt(n)
lower = xbar - t_crit * se
upper = xbar + t_crit * se

print(f"Sample: n={n}, x̄={xbar:.2f}, s={s:.2f}")
print(f"df = {df},  t* = {t_crit:.3f}")
print(f"Standard error (SE): {se:.3f}")
print(f"95% t-CI: ({lower:.2f}, {upper:.2f})")

# Compare to z-CI (which would be narrower — ignores uncertainty in σ)
z_crit = norm.ppf(0.975)
lower_z = xbar - z_crit * se
upper_z = xbar + z_crit * se
print(f"\n95% z-CI (for comparison): ({lower_z:.2f}, {upper_z:.2f})")
print("→ t-CI is wider because it accounts for estimating σ with s.")

In [ ]:
# One-sample t-test
# H0: μ = 75   vs   H1: μ > 75  (one-sided)

mu0    = 75
t_stat = (xbar - mu0) / se
p_val  = 1 - t.cdf(t_stat, df)   # one-sided p-value (right tail)
alpha  = 0.05

print(f"One-sample t-test: H₀: μ = {mu0} vs H₁: μ > {mu0}")
print(f"t statistic: {t_stat:.3f}")
print(f"df = {df}")
print(f"p-value (one-sided): {p_val:.4f}")
print()
if p_val < alpha:
    print(f"p = {p_val:.4f} < α = {alpha} → Reject H₀.")
    print(f"Conclusion: sufficient evidence that μ > {mu0}.")
else:
    print(f"p = {p_val:.4f} ≥ α = {alpha} → Fail to reject H₀.")

---
## Summary

- **Power** = $1 - P(\text{Type II error})$ — the probability of detecting a true effect.
- Power increases with larger $n$, larger effect size, smaller $\sigma$, and larger $\alpha$.
- Sample size formula: $n = \left(\dfrac{z_{1-\alpha} + z_{1-\beta}}{(\mu_1 - \mu_0)/\sigma}\right)^2$.
- A **p-value** is the probability of data at least this extreme *if $H_0$ is true* — not the probability $H_0$ is true.
- When $\sigma$ is unknown, use $t = (\bar{x} - \mu_0)/(s/\sqrt{n})$ with $df = n - 1$.
- The $t$-distribution has heavier tails than the normal; it converges to $N(0,1)$ as $df \to \infty$.
- $t$-CI: $\bar{x} \pm t^*_{n-1}(s/\sqrt{n})$ — always wider than the corresponding $z$-CI.

**Next:** Part 5 – Two-Sample Inference.